In [25]:
import pandas as pd
import os
import textacy
from functools import partial
import spacy

data_directory = os.getcwd()[:-4] + 'original_data/'

In [7]:
df = pd.read_csv(data_directory + 'VideoInfo_ABC.csv')

df['Title'] = df['Title'].str.replace(r'\|.*', '', regex=True)



In [24]:
txt = df.iloc[46]['Title']

txt

'Dems reach agreement on unemployment benefits in COVID-19 relief bill  '

In [44]:
import spacy

# Load the English model
nlp = spacy.load("en_core_web_sm")

# Process the text
doc = nlp("Dems reach agreement on unemployment benefits in COVID-19 relief bill")

# Extract entities
entities = [(ent.text, ent.label_) for ent in doc.ents]
entities


[('Dems', 'NORP'), ('COVID-19', 'ORG')]

In [49]:
import spacy

# Load the English model
nlp = spacy.load("en_core_web_sm")

# Process the text
text = "Dems reach agreement on unemployment benefits in COVID-19 relief bill"
doc = nlp(text)

# Extract entities and meaningful tokens
entities = [ent.text for ent in doc.ents]
meaningful_pos = {"NOUN", "PROPN", "VERB"}
meaningful_tokens = [token.text for token in doc if token.pos_ in meaningful_pos]

# Combine entities and meaningful tokens without repetition
combined_phrases = entities + [token for token in meaningful_tokens if token not in entities]

# Function to generate n-grams from combined phrases without repetition
def generate_ordered_ngrams(doc, n):
    ngrams = []
    for i in range(len(doc) - n + 1):
        ngram = doc[i:i + n]
        if all([token.text in combined_phrases for token in ngram]):
            ngrams.append(" ".join([token.text for token in ngram]))
    return ngrams

# Generate 2-grams, 3-grams, and 4-grams
all_ngrams = []
for n in range(2, 5):
    all_ngrams.extend(generate_ordered_ngrams(doc, n))

# Final tokenized sentence using meaningful phrases and ngrams without repetition
tokenized_sentence = list(set(combined_phrases + all_ngrams))

tokenized_sentence



['Dems reach agreement',
 'Dems reach',
 'COVID-19',
 'agreement',
 'unemployment',
 'unemployment benefits',
 'reach',
 'relief',
 'bill',
 'COVID-19 relief',
 'relief bill',
 'COVID-19 relief bill',
 'Dems',
 'benefits',
 'reach agreement']

In [43]:
from spacy.tokens import Span

nlp = spacy.load("en_core_web_sm")
doc = nlp(txt)

# spacy.displacy.render(doc, style="ent", jupyter=True)


# Extract entities
entities = list(doc.ents)



# Extract bigrams and trigrams using textacy
bigrams = list(textacy.extract.ngrams(doc, n=2))
trigrams = list(textacy.extract.ngrams(doc, n=3))

# Extract nouns and verbs
nouns = [token for token in doc if token.pos_ == "NOUN"]
verbs = [token for token in doc if token.pos_ == "VERB"]

# Create custom spans for n-grams, nouns, and verbs
custom_spans = []

# Add bigrams and trigrams as custom spans
for ngram in bigrams + trigrams:
    custom_spans.append(Span(doc, ngram.start, ngram.end, label="NGRAM"))

# Add nouns as custom spans
for noun in nouns:
    custom_spans.append(Span(doc, noun.i, noun.i + 1, label="NOUN"))

# Add verbs as custom spans
for verb in verbs:
    custom_spans.append(Span(doc, verb.i, verb.i + 1, label="VERB"))

# Add custom spans to the doc
doc.spans["custom"] = custom_spans

# Visualize entities and custom spans
spacy.displacy.render(doc, style="span", jupyter=True, options={"spans_key": "custom"})


# print([token.text for token in doc])
# print([sent.text for sent in doc.sents])

# # Create n-grams using textacy
# ngrams = list(textacy.extract.entities(doc, include_types={"ORG", "GPE", "LOC", "PERSON", "NORP", "FAC", "PRODUCT", "EVENT", "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"}))  # Change n to the desired n-gram length

# # Print n-grams
# print([ngram.text for ngram in ngrams])

In [23]:
corpus = textacy.Corpus('en_core_web_sm', data=txt)

docs_terms = (
        textacy.extract.terms(
            doc,
            # Extract n-grams of varying lengths with specified POS
            ngs=partial(textacy.extract.ngrams, n=(1,2,3,4,5,6), include_pos={"NOUN"}), 
            # Extract specified entity types
            ents=partial(textacy.extract.entities, include_types={"ORG", "GPE", "LOC", "PERSON", "NORP", "FAC", "PRODUCT", "EVENT", "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"}))
        for doc in corpus  # Iterate over each document in the corpus
    )

tokenized_docs = (
        list(textacy.extract.terms_to_strings(doc_terms, by="lemma"))
        for doc_terms in docs_terms
    )

    # Convert the generator to a list
tokenized_words_list = list(tokenized_docs)

tokenized_words_list

[['dem',
  'agreement',
  'unemployment',
  'benefit',
  'covid-19',
  'relief',
  'bill',
  'unemployment benefit',
  'covid-19 relief',
  'relief bill',
  'covid-19 relief bill']]